In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics import roc_curve, auc, roc_auc_score

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

# Grafici
import matplotlib.pyplot as plt
import seaborn as sns

# Keras
import tensorflow as tf
from scikeras.wrappers import KerasClassifier
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.regularizers import l2, l1
from keras.layers import Dropout
from sklearn.model_selection import RandomizedSearchCV
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.models import load_model

# Optuna per l'ottimizzazione
import optuna
from optuna.samplers import TPESampler

In [2]:
####
# DATASET
####
df = pd.read_parquet("df_pulito.parquet")
# df["bmi"] = (df["Physical-Weight"] / (df["Physical-Height"] ** 2)) * 703

# Train Test sii
pciat_cols = [f"PCIAT-PCIAT_{i:02d}" for i in range(1, 21)]
X = df.drop(columns=["sii", "risultati",
            "PCIAT-PCIAT_Total", "id"] + pciat_cols)
y = df["sii"]  # per valutare sii, praticamente classificazione
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=222)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
# per il training
class_weight_dict = dict(enumerate(class_weights))

cv_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=222)

# VALUTAZIONE sii
---

In [3]:
######
# MODELLO NO REGULARIZATION
######

# serve perchè numpy fa le bizze
if hasattr(y_train, 'to_numpy'):
    y_train_np = y_train.to_numpy()
else:
    y_train_np = y_train


def objective_noreg(trial):
    """
    Funzione obiettivo eseguita da Optuna per ogni singolo trial.
    Costruisce, addestra e valuta una configurazione usando la Cross-Validation.
    """
    # Iperparametri
    n_layers = trial.suggest_int('n_layers', 1, 8)
    activation = trial.suggest_categorical(
        'activation', ['relu', 'elu', 'swish'])
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    batch_size = trial.suggest_categorical(
        'batch_size', [512, 1024, 2048, 4096])

    units_per_layer = [trial.suggest_categorical(
        f'n_units_l{i}', [32, 64, 128, 256, 512, 1024, 2048]) for i in range(n_layers)]

    fold_scores = []

    # Loop di Cross-Validation
    for train_idx, val_idx in cv_stratified.split(X_train_scaled, y_train_np):
        X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_tr, y_val = y_train_np[train_idx], y_train_np[val_idx]

        # Costruzione Modello Keras
        model = Sequential()
        model.add(keras.layers.Input(shape=(X_train_scaled.shape[1],)))

        for units in units_per_layer:
            model.add(Dense(units, activation=activation))

        model.add(Dense(4, activation='softmax'))

        # Ottimizzatore e compile
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
        model.compile(
            loss='sparse_categorical_crossentropy',
            optimizer=optimizer
        )

        # Early Stopping
        es = keras.callbacks.EarlyStopping(
            monitor='val_loss',
            min_delta=0,
            patience=20,
            verbose=0,
            restore_best_weights=True
        )

        # Addestramento sul fold
        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=500,
            batch_size=batch_size,
            callbacks=[es],
            class_weight=class_weight_dict,
            verbose=0
        )

        # Valutazione
        y_pred_prob = model.predict(X_val, verbose=0)
        y_pred = np.argmax(y_pred_prob, axis=1)

        score = f1_score(y_val, y_pred, average='macro')
        fold_scores.append(score)

    return np.mean(fold_scores)


# Esecuzione
if __name__ == "__main__":

    storage_name = "sqlite:///optuna_dnn_tuning_noreg.db"

    study_noreg = optuna.create_study(
        direction="maximize",
        storage=storage_name,
        study_name="dnn_tuning_1",
        load_if_exists=True,
        sampler=TPESampler(seed=222)
    )

    study_noreg.optimize(objective_noreg, n_trials=30, n_jobs=1)

    print("\n### CONFIGURAZIONE OTTIMALE ###")
    for key, value in study_noreg.best_trial.params.items():
        print(f" - {key}: {value}")
    print(
        f"\nMiglior F1-Macro Medio (Cross-Validation): {study_noreg.best_value:.4f}")

    # Salvataggio modello
    best_param_noreg = study_noreg.best_params

    final_model_noreg = Sequential()
    final_model_noreg.add(keras.layers.Input(shape=(X_train_scaled.shape[1],)))

    for i in range(best_param_noreg['n_layers']):
        units = best_param_noreg[f'n_units_l{i}']
        final_model_noreg.add(
            Dense(units, activation=best_param_noreg['activation']))

    final_model_noreg.add(Dense(4, activation='softmax'))

    optimizer_final_noreg = keras.optimizers.Adam(
        learning_rate=best_param_noreg['learning_rate'])
    final_model_noreg.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=optimizer_final_noreg,
        metrics=['accuracy']
    )

    es_final = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        min_delta=0,
        patience=20,
        verbose=1,
        restore_best_weights=True
    )

    final_model_noreg.fit(
        X_train_scaled, y_train_np,
        validation_split=0.1,
        epochs=500,
        batch_size=best_param_noreg['batch_size'],
        callbacks=[es_final],
        class_weight=class_weight_dict,
        verbose=0
    )

    # Salvataggio modello
    final_model_noreg.save('best_model_NOREG.keras')
    print("Modello salvato con successo come 'best_model_NOREG.keras'")

    # Valutazione su test
    print("\n### REPORT DI CLASSIFICAZIONE SUL TEST SET ###")
    y_pred_prob = final_model_noreg.predict(X_test_scaled, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)

    print(classification_report(y_test, y_pred))

[I 2026-06-11 12:49:00,320] A new study created in RDB with name: dnn_tuning_1
[I 2026-06-11 12:49:17,892] Trial 0 finished with value: 0.35486242692349335 and parameters: {'n_layers': 5, 'activation': 'elu', 'learning_rate': 0.0001291415114015285, 'batch_size': 1024, 'n_units_l0': 2048, 'n_units_l1': 128, 'n_units_l2': 2048, 'n_units_l3': 32, 'n_units_l4': 32}. Best is trial 0 with value: 0.35486242692349335.
[I 2026-06-11 12:49:28,178] Trial 1 finished with value: 0.2917477475755232 and parameters: {'n_layers': 8, 'activation': 'relu', 'learning_rate': 0.004557965951597541, 'batch_size': 1024, 'n_units_l0': 2048, 'n_units_l1': 2048, 'n_units_l2': 512, 'n_units_l3': 64, 'n_units_l4': 128, 'n_units_l5': 1024, 'n_units_l6': 512, 'n_units_l7': 128}. Best is trial 0 with value: 0.35486242692349335.
[I 2026-06-11 12:49:40,417] Trial 2 finished with value: 0.2425111410913389 and parameters: {'n_layers': 5, 'activation': 'elu', 'learning_rate': 0.015159348035210234, 'batch_size': 1024, 'n_un


### CONFIGURAZIONE OTTIMALE ###
 - n_layers: 3
 - activation: relu
 - learning_rate: 0.0011003636421703547
 - batch_size: 4096
 - n_units_l0: 512
 - n_units_l1: 256
 - n_units_l2: 64

Miglior F1-Macro Medio (Cross-Validation): 0.3721
Restoring model weights from the end of the best epoch: 34.
Epoch 54: early stopping
Modello salvato con successo come 'best_model_NOREG.keras'

### REPORT DI CLASSIFICAZIONE SUL TEST SET ###
              precision    recall  f1-score   support

         0.0       0.84      0.67      0.74      1742
         1.0       0.29      0.42      0.34       450
         2.0       0.24      0.34      0.28       261
         3.0       0.04      0.12      0.06        16

    accuracy                           0.58      2469
   macro avg       0.35      0.39      0.36      2469
weighted avg       0.67      0.58      0.62      2469



In [4]:
######
# MODELLO L1 LASSO
######

if hasattr(y_train, 'to_numpy'):
    y_train_np = y_train.to_numpy()
else:
    y_train_np = y_train


def objective_l1(trial):
    """
    Funzione obiettivo eseguita da Optuna per ogni singolo trial.
    Costruisce, addestra e valuta una configurazione usando la Cross-Validation e la regolarizzazione L1.
    """
    # Iperparametri
    n_layers = trial.suggest_int('n_layers', 1, 8)
    activation = trial.suggest_categorical(
        'activation', ['relu', 'elu', 'swish'])
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    batch_size = trial.suggest_categorical(
        'batch_size', [512, 1024, 2048, 4096])
    l1_lambda = trial.suggest_float('l1_lambda', 1e-5, 1e-2, log=True)

    units_per_layer = [trial.suggest_categorical(
        f'n_units_l{i}', [32, 64, 128, 256, 512, 1024, 2048]) for i in range(n_layers)]

    fold_scores = []

    # Loop di Cross-Validation
    for train_idx, val_idx in cv_stratified.split(X_train_scaled, y_train_np):
        X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_tr, y_val = y_train_np[train_idx], y_train_np[val_idx]

        # Costruzione Modello Keras
        model = Sequential()
        model.add(keras.layers.Input(shape=(X_train_scaled.shape[1],)))

        for units in units_per_layer:
            model.add(Dense(
                units,
                activation=activation,
                kernel_regularizer=keras.regularizers.l1(l1_lambda)
            ))

        model.add(Dense(4, activation='softmax'))

        # Compilazione
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
        model.compile(
            loss='sparse_categorical_crossentropy',
            optimizer=optimizer
        )

        # Early Stopping
        es = keras.callbacks.EarlyStopping(
            monitor='val_loss',
            min_delta=0,
            patience=20,
            verbose=0,
            restore_best_weights=True
        )

        # Addestramento
        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=500,
            batch_size=batch_size,
            callbacks=[es],
            class_weight=class_weight_dict,
            verbose=0
        )

        # Valutazione sul fold di validazione
        y_pred_prob = model.predict(X_val, verbose=0)
        y_pred = np.argmax(y_pred_prob, axis=1)

        score = f1_score(y_val, y_pred, average='macro')
        fold_scores.append(score)

    return np.mean(fold_scores)


# Processo
if __name__ == "__main__":

    storage_name = "sqlite:///optuna_dnn_tuning_l1.db"

    study_l1 = optuna.create_study(
        direction="maximize",
        storage=storage_name,
        study_name="dnn_l1_tuning_1",
        load_if_exists=True,
        sampler=TPESampler(seed=222)
    )

    study_l1.optimize(objective_l1, n_trials=30, n_jobs=1)

    print("\n### CONFIGURAZIONE OTTIMALE ###")
    for key, value in study_l1.best_trial.params.items():
        print(f" - {key}: {value}")
    print(
        f"\nMiglior F1-Macro Medio (Cross-Validation L1): {study_l1.best_value:.4f}")

    # Addestramento e Salvataggio
    best_param_l1 = study_l1.best_params

    final_model_l1 = Sequential()
    final_model_l1.add(keras.layers.Input(shape=(X_train_scaled.shape[1],)))

    for i in range(best_param_l1['n_layers']):
        units = best_param_l1[f'n_units_l{i}']
        final_model_l1.add(Dense(
            units,
            activation=best_param_l1['activation'],
            kernel_regularizer=keras.regularizers.l1(
                best_param_l1['l1_lambda'])
        ))

    final_model_l1.add(Dense(4, activation='softmax'))

    # Compilazione
    optimizer_final_l1 = keras.optimizers.Adam(
        learning_rate=best_param_l1['learning_rate'])
    final_model_l1.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=optimizer_final_l1,
        metrics=['accuracy']
    )

    # Early Stopping
    es_final = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        min_delta=0,
        patience=20,
        verbose=1,
        restore_best_weights=True
    )

    # Addestramento finale
    final_model_l1.fit(
        X_train_scaled, y_train_np,
        validation_split=0.1,
        epochs=500,
        batch_size=best_param_l1['batch_size'],
        callbacks=[es_final],
        class_weight=class_weight_dict,
        verbose=0
    )

    # Salvataggio effettivo
    final_model_l1.save('best_model_L1.keras')

    # Valutazione su Test
    y_pred_prob = final_model_l1.predict(X_test_scaled, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)

    print(classification_report(y_test, y_pred))

[I 2026-06-11 12:54:00,690] A new study created in RDB with name: dnn_l1_tuning_1
[I 2026-06-11 12:55:12,037] Trial 0 finished with value: 0.3499916416586874 and parameters: {'n_layers': 5, 'activation': 'elu', 'learning_rate': 0.0001291415114015285, 'batch_size': 1024, 'l1_lambda': 7.285701681802387e-05, 'n_units_l0': 1024, 'n_units_l1': 64, 'n_units_l2': 2048, 'n_units_l3': 2048, 'n_units_l4': 2048}. Best is trial 0 with value: 0.3499916416586874.
[I 2026-06-11 12:55:27,161] Trial 1 finished with value: 0.06302929219061384 and parameters: {'n_layers': 8, 'activation': 'swish', 'learning_rate': 0.004499862776019602, 'batch_size': 512, 'l1_lambda': 0.00014551819467846668, 'n_units_l0': 512, 'n_units_l1': 512, 'n_units_l2': 2048, 'n_units_l3': 128, 'n_units_l4': 32, 'n_units_l5': 256, 'n_units_l6': 128, 'n_units_l7': 32}. Best is trial 0 with value: 0.3499916416586874.
[I 2026-06-11 12:55:56,282] Trial 2 finished with value: 0.2639273140036398 and parameters: {'n_layers': 6, 'activation


### CONFIGURAZIONE OTTIMALE ###
 - n_layers: 4
 - activation: swish
 - learning_rate: 0.0011205780974154582
 - batch_size: 512
 - l1_lambda: 5.9094864819845464e-05
 - n_units_l0: 2048
 - n_units_l1: 128
 - n_units_l2: 1024
 - n_units_l3: 512

Miglior F1-Macro Medio (Cross-Validation L1): 0.3757
Restoring model weights from the end of the best epoch: 41.
Epoch 61: early stopping
              precision    recall  f1-score   support

         0.0       0.83      0.68      0.75      1742
         1.0       0.30      0.31      0.30       450
         2.0       0.25      0.43      0.31       261
         3.0       0.02      0.12      0.03        16

    accuracy                           0.58      2469
   macro avg       0.35      0.39      0.35      2469
weighted avg       0.67      0.58      0.62      2469



In [5]:
######
# MODELLO L2 RIDGE
######

if hasattr(y_train, 'to_numpy'):
    y_train_np = y_train.to_numpy()
else:
    y_train_np = y_train


def objective_l2(trial):
    """
    Funzione obiettivo eseguita da Optuna per ogni singolo trial.
    Costruisce, addestra e valuta una configurazione usando la Cross-Validation e la regolarizzazione L2.
    """
    # Iperparametri
    n_layers = trial.suggest_int('n_layers', 1, 8)
    activation = trial.suggest_categorical(
        'activation', ['relu', 'elu', 'swish'])
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    batch_size = trial.suggest_categorical(
        'batch_size', [512, 1024, 2048, 4096])
    l2_lambda = trial.suggest_float('l2_lambda', 1e-5, 1e-2, log=True)

    units_per_layer = [trial.suggest_categorical(
        f'n_units_l{i}', [32, 64, 128, 256, 512, 1024, 2048]) for i in range(n_layers)]

    fold_scores = []

    # Loop di Cross-Validation
    for train_idx, val_idx in cv_stratified.split(X_train_scaled, y_train_np):
        X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_tr, y_val = y_train_np[train_idx], y_train_np[val_idx]

        # Costruzione Modello Keras
        model = Sequential()
        model.add(keras.layers.Input(shape=(X_train_scaled.shape[1],)))

        for units in units_per_layer:
            model.add(Dense(
                units,
                activation=activation,
                kernel_regularizer=keras.regularizers.l2(l2_lambda)
            ))

        model.add(Dense(4, activation='softmax'))

        # Compilazione
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
        model.compile(
            loss='sparse_categorical_crossentropy',
            optimizer=optimizer
        )

        # Early Stopping
        es = keras.callbacks.EarlyStopping(
            monitor='val_loss',
            min_delta=0,
            patience=20,
            verbose=0,
            restore_best_weights=True
        )

        # Addestramento
        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=500,
            batch_size=batch_size,
            callbacks=[es],
            class_weight=class_weight_dict,
            verbose=0
        )

        # Valutazione
        y_pred_prob = model.predict(X_val, verbose=0)
        y_pred = np.argmax(y_pred_prob, axis=1)

        score = f1_score(y_val, y_pred, average='macro')
        fold_scores.append(score)

    return np.mean(fold_scores)


# Processo
if __name__ == "__main__":

    storage_name = "sqlite:///optuna_dnn_tuning_l2.db"

    study_l2 = optuna.create_study(
        direction="maximize",
        storage=storage_name,
        study_name="dnn_l2_tuning_1",
        load_if_exists=True,
        sampler=TPESampler(seed=222)
    )

    study_l2.optimize(objective_l2, n_trials=30, n_jobs=1)

    print("\n### CONFIGURAZIONE OTTIMALE ###")
    for key, value in study_l2.best_trial.params.items():
        print(f" - {key}: {value}")
    print(
        f"\nMiglior F1-Macro Medio (Cross-Validation L2): {study_l2.best_value:.4f}")

    # Addestramento e Salvataggio
    best_param_l2 = study_l2.best_params
    final_model_l2 = Sequential()
    final_model_l2.add(keras.layers.Input(shape=(X_train_scaled.shape[1],)))

    for i in range(best_param_l2['n_layers']):
        units = best_param_l2[f'n_units_l{i}']
        final_model_l2.add(Dense(
            units,
            activation=best_param_l2['activation'],
            kernel_regularizer=keras.regularizers.l2(
                best_param_l2['l2_lambda'])
        ))

    final_model_l2.add(Dense(4, activation='softmax'))

    # Compilazione
    optimizer_final_l2 = keras.optimizers.Adam(
        learning_rate=best_param_l2['learning_rate'])
    final_model_l2.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=optimizer_final_l2,
        metrics=['accuracy']
    )

    es_final = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        min_delta=0,
        patience=20,
        verbose=1,
        restore_best_weights=True
    )

    # Addestramento finale
    final_model_l2.fit(
        X_train_scaled, y_train_np,
        validation_split=0.1,
        epochs=500,
        batch_size=best_param_l2['batch_size'],
        callbacks=[es_final],
        class_weight=class_weight_dict,
        verbose=0
    )

    # Salvataggio
    final_model_l2.save('best_model_L2.keras')

    y_pred_prob = final_model_l2.predict(X_test_scaled, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)

    print(classification_report(y_test, y_pred))

[I 2026-06-11 13:05:41,993] A new study created in RDB with name: dnn_l2_tuning_1
[I 2026-06-11 13:06:06,741] Trial 0 finished with value: 0.35826863878884946 and parameters: {'n_layers': 5, 'activation': 'elu', 'learning_rate': 0.0001291415114015285, 'batch_size': 1024, 'l2_lambda': 7.285701681802387e-05, 'n_units_l0': 1024, 'n_units_l1': 64, 'n_units_l2': 2048, 'n_units_l3': 2048, 'n_units_l4': 2048}. Best is trial 0 with value: 0.35826863878884946.
[I 2026-06-11 13:06:22,151] Trial 1 finished with value: 0.3574993779834567 and parameters: {'n_layers': 8, 'activation': 'swish', 'learning_rate': 0.004499862776019602, 'batch_size': 512, 'l2_lambda': 0.00014551819467846668, 'n_units_l0': 512, 'n_units_l1': 512, 'n_units_l2': 2048, 'n_units_l3': 128, 'n_units_l4': 32, 'n_units_l5': 256, 'n_units_l6': 128, 'n_units_l7': 32}. Best is trial 0 with value: 0.35826863878884946.
[I 2026-06-11 13:06:59,575] Trial 2 finished with value: 0.26534730964096315 and parameters: {'n_layers': 6, 'activat


### CONFIGURAZIONE OTTIMALE ###
 - n_layers: 4
 - activation: relu
 - learning_rate: 0.0006542212194440559
 - batch_size: 2048
 - l2_lambda: 0.008985674143939049
 - n_units_l0: 2048
 - n_units_l1: 1024
 - n_units_l2: 512
 - n_units_l3: 512

Miglior F1-Macro Medio (Cross-Validation L2): 0.3827
Restoring model weights from the end of the best epoch: 58.
Epoch 78: early stopping
              precision    recall  f1-score   support

         0.0       0.84      0.75      0.79      1742
         1.0       0.31      0.46      0.37       450
         2.0       0.30      0.24      0.27       261
         3.0       0.03      0.06      0.04        16

    accuracy                           0.64      2469
   macro avg       0.37      0.38      0.37      2469
weighted avg       0.68      0.64      0.66      2469



In [6]:
######
# MODELLO DROPOUT
######

if hasattr(y_train, 'to_numpy'):
    y_train_np = y_train.to_numpy()
else:
    y_train_np = y_train


def objective_drop(trial):
    """
    Funzione obiettivo eseguita da Optuna per ogni singolo trial.
    Costruisce, addestra e valuta una configurazione usando la Cross-Validation e il Dropout.
    """
    # Iperparametri
    n_layers = trial.suggest_int('n_layers', 1, 8)
    activation = trial.suggest_categorical(
        'activation', ['relu', 'elu', 'swish'])
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    batch_size = trial.suggest_categorical(
        'batch_size', [512, 1024, 2048, 4096])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5, step=0.05)

    units_per_layer = [trial.suggest_categorical(
        f'n_units_l{i}', [32, 64, 128, 256, 512, 1024, 2048]) for i in range(n_layers)]

    fold_scores = []

    # Loop di Cross-Validation
    for train_idx, val_idx in cv_stratified.split(X_train_scaled, y_train_np):
        X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_tr, y_val = y_train_np[train_idx], y_train_np[val_idx]

        # Costruzione Modello Keras
        model = Sequential()
        model.add(keras.layers.Input(shape=(X_train_scaled.shape[1],)))

        for units in units_per_layer:
            model.add(Dense(units, activation=activation))
            model.add(Dropout(dropout_rate))

        model.add(Dense(4, activation='softmax'))

        # Compilazione
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
        model.compile(
            loss='sparse_categorical_crossentropy',
            optimizer=optimizer
        )

        # Early Stopping
        es = keras.callbacks.EarlyStopping(
            monitor='val_loss',
            min_delta=0,
            patience=20,
            verbose=0,
            restore_best_weights=True
        )

        # Addestramento sul fold
        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=500,
            batch_size=batch_size,
            callbacks=[es],
            class_weight=class_weight_dict,
            verbose=0
        )

        # Valutazione sul fold di validazione
        y_pred_prob = model.predict(X_val, verbose=0)
        y_pred = np.argmax(y_pred_prob, axis=1)

        score = f1_score(y_val, y_pred, average='macro')
        fold_scores.append(score)

    return np.mean(fold_scores)


# Processo
if __name__ == "__main__":

    storage_name = "sqlite:///optuna_dnn_tuning_dropout.db"

    study_dropout = optuna.create_study(
        direction="maximize",
        storage=storage_name,
        study_name="dnn_dropout_tuning_1",
        load_if_exists=True,
        sampler=TPESampler(seed=222)
    )

    study_dropout.optimize(objective_drop, n_trials=30, n_jobs=1)

    print("\n### CONFIGURAZIONE OTTIMALE ###")
    for key, value in study_dropout.best_trial.params.items():
        print(f" - {key}: {value}")
    print(
        f"\nMiglior F1-Macro Medio (Cross-Validation Dropout): {study_dropout.best_value:.4f}")

    # Addestramento e Salvataggio
    best_param_dropout = study_dropout.best_params

    final_model_dropout = Sequential()
    final_model_dropout.add(keras.layers.Input(
        shape=(X_train_scaled.shape[1],)))

    for i in range(best_param_dropout['n_layers']):
        units = best_param_dropout[f'n_units_l{i}']
        final_model_dropout.add(
            Dense(units, activation=best_param_dropout['activation']))
        final_model_dropout.add(Dropout(best_param_dropout['dropout_rate']))

    final_model_dropout.add(Dense(4, activation='softmax'))

    # Compilazione
    optimizer_final_drop = keras.optimizers.Adam(
        learning_rate=best_param_dropout['learning_rate'])
    final_model_dropout.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=optimizer_final_drop,
        metrics=['accuracy']
    )

    # Early Stopping
    es_final = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        min_delta=0,
        patience=20,
        verbose=1,
        restore_best_weights=True
    )

    # Addestramento finale
    final_model_dropout.fit(
        X_train_scaled, y_train_np,
        validation_split=0.1,
        epochs=500,
        batch_size=best_param_dropout['batch_size'],
        callbacks=[es_final],
        class_weight=class_weight_dict,
        verbose=0
    )

    # Salvataggio
    final_model_dropout.save('best_model_DROPOUT.keras')

    # Valutazione su test
    y_pred_prob = final_model_dropout.predict(X_test_scaled, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)

    print(classification_report(y_test, y_pred))

[I 2026-06-11 13:13:24,738] A new study created in RDB with name: dnn_dropout_tuning_1
[I 2026-06-11 13:13:47,774] Trial 0 finished with value: 0.33513272332008026 and parameters: {'n_layers': 5, 'activation': 'elu', 'learning_rate': 0.0001291415114015285, 'batch_size': 1024, 'dropout_rate': 0.2, 'n_units_l0': 1024, 'n_units_l1': 64, 'n_units_l2': 2048, 'n_units_l3': 2048, 'n_units_l4': 2048}. Best is trial 0 with value: 0.33513272332008026.
[I 2026-06-11 13:14:10,751] Trial 1 finished with value: 0.3209984765333619 and parameters: {'n_layers': 8, 'activation': 'swish', 'learning_rate': 0.004499862776019602, 'batch_size': 512, 'dropout_rate': 0.25, 'n_units_l0': 512, 'n_units_l1': 512, 'n_units_l2': 2048, 'n_units_l3': 128, 'n_units_l4': 32, 'n_units_l5': 256, 'n_units_l6': 128, 'n_units_l7': 32}. Best is trial 0 with value: 0.33513272332008026.
[I 2026-06-11 13:14:21,504] Trial 2 finished with value: 0.2032667027836645 and parameters: {'n_layers': 6, 'activation': 'elu', 'learning_rat


### CONFIGURAZIONE OTTIMALE ###
 - n_layers: 4
 - activation: swish
 - learning_rate: 0.0015241716923182736
 - batch_size: 512
 - dropout_rate: 0.35
 - n_units_l0: 2048
 - n_units_l1: 256
 - n_units_l2: 128
 - n_units_l3: 256

Miglior F1-Macro Medio (Cross-Validation Dropout): 0.3776
Restoring model weights from the end of the best epoch: 16.
Epoch 36: early stopping
              precision    recall  f1-score   support

         0.0       0.82      0.64      0.72      1742
         1.0       0.28      0.34      0.31       450
         2.0       0.24      0.39      0.29       261
         3.0       0.01      0.12      0.03        16

    accuracy                           0.56      2469
   macro avg       0.34      0.37      0.34      2469
weighted avg       0.66      0.56      0.60      2469



In [7]:
####
# COMPARISON MODELLI
####

# Caricamento modelli migliori
best_NOREG_model = load_model('best_model_NOREG.keras')
best_L1_model = load_model('best_model_L1.keras')
best_L2_model = load_model('best_model_L2.keras')
best_DROPOUT_model = load_model('best_model_DROPOUT.keras')

# Valutazioni di Keras
loss_NOREG, acc_NOREG = best_NOREG_model.evaluate(
    X_test_scaled, y_test, verbose=0)
loss_L1, acc_L1 = best_L1_model.evaluate(X_test_scaled, y_test, verbose=0)
loss_L2, acc_L2 = best_L2_model.evaluate(X_test_scaled, y_test, verbose=0)
loss_DROPOUT, acc_DROPOUT = best_DROPOUT_model.evaluate(
    X_test_scaled, y_test, verbose=0)

# Predizioni
pred_NOREG = np.argmax(best_NOREG_model.predict(
    X_test_scaled, verbose=0), axis=1)
pred_L1 = np.argmax(best_L1_model.predict(X_test_scaled, verbose=0), axis=1)
pred_L2 = np.argmax(best_L2_model.predict(X_test_scaled, verbose=0), axis=1)
pred_DROPOUT = np.argmax(best_DROPOUT_model.predict(
    X_test_scaled, verbose=0), axis=1)

# Calcolo F1-Score Macro
f1_NOREG = f1_score(y_test, pred_NOREG, average='macro')
f1_L1 = f1_score(y_test, pred_L1, average='macro')
f1_L2 = f1_score(y_test, pred_L2, average='macro')
f1_DROPOUT = f1_score(y_test, pred_DROPOUT, average='macro')

# Stampa finale
print("### COMPARAZIONE FINALE MODELLI DNN ###\n")
print(f"{'Modello':<25} | {'Loss':<8} | {'Accuracy':<10} | {'F1-Macro':<10}")
print("-" * 65)
print(f"{'No Regolarizzazione':<25} | {loss_NOREG:<8.4f} | {acc_NOREG:<10.4f} | {f1_NOREG:<10.4f}")
print(f"{'L1 (Lasso)':<25} | {loss_L1:<8.4f} | {acc_L1:<10.4f} | {f1_L1:<10.4f}")
print(f"{'L2 (Ridge)':<25} | {loss_L2:<8.4f} | {acc_L2:<10.4f} | {f1_L2:<10.4f}")
print(f"{'Dropout':<25} | {loss_DROPOUT:<8.4f} | {acc_DROPOUT:<10.4f} | {f1_DROPOUT:<10.4f}")

### COMPARAZIONE FINALE MODELLI DNN ###

Modello                   | Loss     | Accuracy   | F1-Macro  
-----------------------------------------------------------------
No Regolarizzazione       | 0.9845   | 0.5832     | 0.3564    
L1 (Lasso)                | 1.2252   | 0.5828     | 0.3488    
L2 (Ridge)                | 1.3151   | 0.6403     | 0.3687    
Dropout                   | 0.9675   | 0.5569     | 0.3366    


## Logistica
---

In [9]:
# Logistica

# Inizializzazione del modello base
logistica_base = LogisticRegression(
    random_state=222,
    class_weight='balanced',
    multi_class='multinomial',
    max_iter=1000,
    solver='lbfgs',  # algoritmo specializzato per l2; prima usavo saga
    penalty='l2',  # facendo delle prove, l2 è meglio di l1
    tol=1e-3
)

# Iperparametri
parametri_log = {
    # regolarizzazione della logistica
    'C': [0.0001, 0.001, 0.01, 0.1, 1, 10, 100]
}

# Grid Search
gs_logistica = GridSearchCV(
    estimator=logistica_base,
    param_grid=parametri_log,
    scoring='f1_macro',
    cv=10,
    verbose=1,
    n_jobs=1
)

# Training
gs_logistica.fit(X_train_scaled, y_train)

# Risultati
best_logistica = gs_logistica.best_estimator_

print("### CONFIGURAZIONE OTTIMALE LOGISTICA ###")
for parametro, valore in gs_logistica.best_params_.items():
    print(f" - {parametro}: {valore}")

print("\n### REPORT DI CLASSIFICAZIONE (TEST SET) ###")
y_pred_log = best_logistica.predict(X_test_scaled)
print(classification_report(y_test, y_pred_log))

Fitting 10 folds for each of 7 candidates, totalling 70 fits


c:\Users\Riccardo\anaconda3\envs\tf-directml\lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Riccardo\anaconda3\envs\tf-directml\lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Riccardo\anaconda3\envs\tf-directml\lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Riccardo\anaconda3\envs\tf-directml\lib\site-packages\sklearn\linea

### CONFIGURAZIONE OTTIMALE LOGISTICA ###
 - C: 0.001

### REPORT DI CLASSIFICAZIONE (TEST SET) ###
              precision    recall  f1-score   support

         0.0       0.83      0.64      0.72      1742
         1.0       0.28      0.22      0.25       450
         2.0       0.22      0.35      0.27       261
         3.0       0.01      0.31      0.03        16

    accuracy                           0.53      2469
   macro avg       0.34      0.38      0.32      2469
weighted avg       0.66      0.53      0.58      2469



c:\Users\Riccardo\anaconda3\envs\tf-directml\lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Riccardo\anaconda3\envs\tf-directml\lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Riccardo\anaconda3\envs\tf-directml\lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Riccardo\anaconda3\envs\tf-directml\lib\site-packages\sklearn\linea